# Indicator 7 — Support Services Consolidation

Single notebook replacing the seven per-source notebooks (`heart`, `bg`, `coursera_extraction`,
`lxcheckins`, `placements`, `sap`, `xpl`). Produces one consolidated `indicator_7_data.csv`.

## How to run (monthly)

1. Edit **Section 1 (Configuration)** only: source file paths, reporting period, and any
   email corrections or exclusions carried over from last month's review.
2. `Kernel > Restart & Run All`.
3. If the notebook **halts**, read the validation table printed at the point of failure.
   Every failure message states what is wrong and which CONFIG entry to update. No other
   cells should need editing.
4. On success, Section 10 prints a per-source summary and writes the output file **once**
   (overwrite, with header). There is no append mode, so re-running is always safe.

## Changes vs the old per-source notebooks

- `month_of_service_accessed` year is derived from the service date (the old xpl,
  lxcheckins and placements notebooks hardcoded `' 2025'`).
- The output file is written once at the end. The old flow depended on running
  lxcheckins first (overwrite) and appending the rest headerless; running any notebook
  twice duplicated rows.
- Manual email fixes and confirmed-unreportable learners live in CONFIG, not in
  anonymous cells mid-notebook.
- Unmatched emails halt the run with a review table instead of being dropped silently.


In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

## 1. Configuration

**This is the only section that should change month to month.**

In [ ]:
# ------------------------------------------------------------------
# Reporting window (inclusive). Rows outside it are flagged, and
# sources with filter_to_period=True are trimmed to it.
# ------------------------------------------------------------------
REPORTING_PERIOD_START = "2026-06-01"      # <-- MONTHLY EDIT
REPORTING_PERIOD_END   = "2026-06-30"      # <-- MONTHLY EDIT

# Data root on this machine -- see the repo README for the expected layout
BASE      = "/Users/markngendo/Documents/Umuzi/Reporting"
MONTH_DIR = f"{BASE}/Monthly IDC/June (2026)"   # <-- MONTHLY EDIT (match folder naming on disk)

SRC  = f"{MONTH_DIR}/Source Datasets"
SINK = f"{MONTH_DIR}/Sink Datasets"

# Enrichment dataset exported from the database
RICH_PATH = "/Users/markngendo/Documents/Umuzi/Reporting/Improved IDC/Categories/Support/db_fields.csv"

# Consolidated output. Overwritten in full on every successful run.
SINK_PATH = f"{SINK}/indicator_7_data.csv"


# ------------------------------------------------------------------
# Per-source inputs. Set enabled=False to skip a source this month.
# filter_to_period=True trims rows to the reporting window (use for
# exports that contain full history, e.g. Coursera). When False, out-
# of-window rows are only reported by validation, not removed.
# ------------------------------------------------------------------
SOURCES = {
    "heart": {
        "enabled": True,
        "path": f"{SRC}/May 2026_Support Service Template - Community_Heart Team - Copy of Support Service - Connection Session.csv",
        "filter_to_period": False,
    },
    "bg": {
        "enabled": True,
        "path": f"{SRC}/Be Green  Support Service Template - April 2026 - Begreen - General Support Service.csv",
        "constant_cohort": "BeGreen LB3",
        "filter_to_period": False,
    },
    "coursera": {
        "enabled": True,
        "path": "/Users/markngendo/Downloads/LearnerActivity umuzi - CourseraEnterpriseExport 2026-05-25 07-14-31 UTC/CourseActivity umuzi - CourseraEnterpriseExport 2026-05-25 07-14-31 UTC.csv",
        "filter_to_period": True,   # full-history export; must be trimmed
    },
    "lxcheckins": {
        "enabled": True,
        "path": f"{SRC}/May 2026 _ Support Service_ LX Coach Check Ins [Muzi].xlsx",
        "filter_to_period": False,
    },
    "placements": {
        "enabled": True,
        "path": "/Users/markngendo/Documents/Umuzi/Reporting/Monthly IDC/March(2026)/Source Datasets/Copy of March 2026 - Support Service  - Launch Lab_Anchen.xlsx",
        "filter_to_period": False,
    },
    "sap": {
        "enabled": True,
        "path": f"{SRC}/April 2026 - SAP Support Service Template .xlsx",
        "filter_to_period": True,
    },
    "xpl": {
        "enabled": True,
        "path": f"{SRC}/Y2 XPL Support Service - _continuous updating_ - _Natalie Jorgensen_.xlsx",
        "filter_to_period": False,
    },
}

# ------------------------------------------------------------------
# Known-bad emails -> corrections, applied to ALL sources before any
# matching. Add entries here when the unmatched-email review (Section
# 6) identifies a typo or a personal address with a known canonical
# equivalent.
# ------------------------------------------------------------------
EMAIL_CORRECTIONS = {
    # left empty in the repository to not leak personal information; Ask Mark Ngendo for this
}

# ------------------------------------------------------------------
# Emails confirmed NOT reportable for IDC (verified non-SA citizens,
# staff, or absent from the database after investigation). Rows with
# these emails are removed with a printed count. Only add an email
# here after it has actually been checked.
# ------------------------------------------------------------------
KNOWN_UNREPORTABLE_EMAILS = {
    # left empty in the repository to not leak personal information; Ask Mark Ngendo for this
}

# ------------------------------------------------------------------
# What to do with emails that STILL cannot be matched to the database
# after corrections and exclusions:
#   "halt" - stop the run and print a review table (recommended)
#   "drop" - drop the rows, but print the same table prominently
# ------------------------------------------------------------------
UNMATCHED_EMAIL_POLICY = "halt"

# Coursera-specific: programmes whose learners are employed and are
# therefore excluded from indicator 7.
COURSERA_DROP_SERVICES = [
    "Capitec - E12 Java",
    "Capitec E10 - Data Analytics",
    "Umuzi- Main - Staff Learning Program",
    "Capitec E10",
    "Capitec E10 - Java",
    "Capitec E9 - Java Development",
]

# Coursera-specific: Program Name -> reporting cohort. When the run
# halts on "unmapped Coursera programmes", add the new programme here.
COURSERA_COHORT_MAP = {
    "BE Green": "BeGreen Cohort",
    "BE GREEN LB2": "BeGreen Cohort",
    "SAP Developer Associate C2": "SAP_Umuzi_Y2",
    "SAP General Tech Consultant  C2": "SAP_Umuzi_Y2",
    "SAP Human Capital Management C2": "SAP_Umuzi_Y2",
    "VML (C50)": "XH-AUXUI-Aug-25",
    "Capitec E9 - Java Development": "e9_java",
    "WesBank  Project Management": "SL-PM-Wesbank-Nov-25",
    "XPL Block 3 - Accelerating Success  - XPL_B3_AS": "XA-Nov-25",
    "Capitec E10": "e10_da",
    "Capitec E10 - Java": "e10_da",
    "Experience Lab Advanced Web Development XA1": "XH-AWD-Aug-25",
    "Advanced WD: XH-AWD-Feb26": "XH-AWD-Feb-26",
    "Accenture: Skill Advancement": "Accenture: Skill Advancement",
    "Year 3: Support Role(Generalist )": "SAP_Umuzi_Y3_Jan26",
    "Year 3: Human Capital Management": "SAP_Umuzi_Y3_Jan26",
    "Year 3: SAP Developer Associate": "SAP_Umuzi_Y3_Jan26",
    "BEGREEN LB 3: Business Acceleration": "BeGreen Cohort",
    "Foundational WebDevelopment: SL-FWD-Feb26 (C51_BBD)": "SL-FWD-Feb26",
    "Advanced Project management": "SAP Umuzi",
    "SAP Bootcamp 2026": "SAP Bootcamp 2026",
}

## 2. Helpers and validation framework

`check()` records a named validation. Failures print their diagnostic detail immediately.
`halt_if_failed()` stops the run with a plain-language summary of every failed
error-level check, so a reviewer sees the full picture in one place instead of
hitting failures one at a time.

In [ ]:
REPORTING_PERIOD_START = pd.to_datetime(REPORTING_PERIOD_START)
REPORTING_PERIOD_END   = pd.to_datetime(REPORTING_PERIOD_END)

FINAL_COLUMNS = [
    "learner_id", "application_id", "umuzi_email",
    "first_name", "last_name", "cohort_name", "cellphone_number",
    "id_number", "gender", "date_of_birth", "age_service_accessed", "age_range", "race",
    "residential_area_type", "has_disability_or_differently_abled",
    "application_date", "nearest_metro", "province", "date_service_accessed",
    "month_of_service_accessed", "service_used", "service_name",
]

# Contract every loader must satisfy
LOADER_COLUMNS = {"umuzi_email", "cohort_name", "service_used",
                  "service_name", "date_service_accessed"}


def parse_date(val):
    """Parse the date formats that appear across the support service templates."""
    formats = [
        "%Y-%m-%d %H:%M:%S.%f",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d",
        "%d/%m/%Y %H:%M:%S.%f",
        "%d/%m/%Y %H:%M:%S",
        "%d/%m/%Y",
    ]
    for fmt in formats:
        try:
            return pd.to_datetime(val, format=fmt)
        except Exception:
            continue
    try:
        return pd.to_datetime(val)   # last resort: let pandas infer
    except Exception:
        return pd.NaT


def normalize_emails(series):
    return series.str.strip().str.lower()


class Checks:
    """Collects validation results and prints failures with actionable detail."""

    def __init__(self):
        self.rows = []

    def add(self, stage, name, passed, detail="", offenders=None, level="error"):
        self.rows.append({
            "stage": stage, "check": name,
            "status": "PASS" if passed else ("FAIL" if level == "error" else "WARN"),
            "level": level, "detail": "" if passed else detail,
        })
        if not passed:
            tag = "FAIL" if level == "error" else "WARN"
            print(f"[{tag}] {stage} :: {name}")
            if detail:
                print(f"       {detail}")
            if offenders is not None and len(offenders):
                display(offenders)

    def summary(self):
        return pd.DataFrame(self.rows)

    def halt_if_failed(self, stage=None):
        df = self.summary()
        if stage is not None:
            df = df[df["stage"] == stage]
        failed = df[df["status"] == "FAIL"]
        if not failed.empty:
            display(failed[["stage", "check", "detail"]])
            raise AssertionError(
                f"{len(failed)} validation failure(s). Review the table above; each "
                "detail line names the CONFIG entry or source file to fix. Then "
                "Restart & Run All."
            )


checks = Checks()

## 3. Enrichment data (`db_fields.csv`)

Both email columns are normalized once here, so all downstream matching is
case- and whitespace-insensitive. Validations guard the assumptions every merge
below relies on: unique `umuzi_email` (otherwise merges fan out) and a clean
personal-email to umuzi-email mapping.

In [ ]:
rich = pd.read_csv(
    RICH_PATH,
    dtype={"cellphone_number": "str", "id_number": "str"},
)

rich["umuzi_email"] = normalize_emails(rich["umuzi_email"])
rich["email"] = normalize_emails(rich["email"])

required_rich = {"email", "umuzi_email", "learner_id", "date_of_birth", "id_number"}
missing = required_rich - set(rich.columns)
checks.add("rich", "required columns present", not missing,
           f"db_fields.csv is missing columns: {sorted(missing)}. Re-export it.")

dup_umuzi = rich[rich.duplicated(subset=["umuzi_email"], keep=False)]
checks.add("rich", "umuzi_email is unique", dup_umuzi.empty,
           "Duplicate umuzi_email values in db_fields.csv will duplicate rows on "
           "merge. Deduplicate the export before continuing.",
           offenders=dup_umuzi.sort_values("umuzi_email"))

dup_personal = rich[rich["email"].notna() & rich.duplicated(subset=["email"], keep=False)]
checks.add("rich", "personal email maps to a single umuzi_email", dup_personal.empty,
           "The same personal email maps to multiple umuzi emails; the remap "
           "below would be ambiguous.",
           offenders=dup_personal.sort_values("email"), level="warn")

checks.halt_if_failed("rich")

# personal email -> canonical umuzi email
emailMap = rich.set_index("email")["umuzi_email"]
print(f"db_fields loaded: {rich.shape[0]} learners")

## 4. Source loaders

One function per source. Each returns a frame with exactly the loader contract
columns (`umuzi_email`, `cohort_name`, `service_used`, `service_name`,
`date_service_accessed`) and contains only the cleaning that is genuinely
specific to that template. Everything shared — email canonicalization,
enrichment, derived fields, validation — happens once, downstream.

In [ ]:
def load_heart(cfg):
    """Heart team connection sessions (CSV). Emails arrive comma-separated."""
    return (
        pd.read_csv(
            cfg["path"], usecols=[2, 3, 5, 6, 7], skiprows=2,
            names=["umuzi_email", "cohort_name", "service_used",
                   "service_name", "date_service_accessed"],
        )
        .assign(umuzi_email=lambda d: d["umuzi_email"].str.split(r",\s*"))
        .explode("umuzi_email", ignore_index=True)
        .assign(date_service_accessed=lambda d: d["date_service_accessed"].map(parse_date))
        .dropna(subset=["umuzi_email"])
    )

In [ ]:
def load_bg(cfg):
    """Be Green general support (CSV). Both emails and service names arrive
    comma-separated; cohort is constant for the whole file (see CONFIG)."""
    return (
        pd.read_csv(
            cfg["path"], usecols=[2, 4, 5, 6], skiprows=3,
            names=["umuzi_email", "service_used", "service_name", "date_service_accessed"],
        )
        .assign(
            service_name=lambda d: d["service_name"].str.split(r",\s*"),
            umuzi_email=lambda d: d["umuzi_email"].str.split(r",\s*"),
        )
        .explode("service_name", ignore_index=True)
        .explode("umuzi_email", ignore_index=True)
        .assign(
            service_name=lambda d: d["service_name"].str.strip(),
            date_service_accessed=lambda d: d["date_service_accessed"].map(parse_date),
            cohort_name=cfg["constant_cohort"],
        )
        .dropna(subset=["umuzi_email"])
    )

In [ ]:
def load_coursera(cfg):
    """Coursera enterprise course-activity export (CSV, full history).

    Only completed courses count. The export truncates some timestamps to
    minutes; those get :00 seconds appended so parse_date sees one format.
    Cohort is resolved from Program Name via COURSERA_COHORT_MAP, with two
    pattern-based overrides (SAP pathways, Accenture)."""
    df = (
        pd.read_csv(cfg["path"])
        .pipe(lambda d: d.loc[d["Completion Time"].notna()])
        .loc[:, ["Name", "Email", "Course", "Program Name", "Completion Time"]]
        .rename(columns={
            "Email": "umuzi_email",
            "Course": "service_name",
            "Program Name": "service_used",
            "Completion Time": "date_service_accessed",
        })
        .assign(
            date_service_accessed=lambda d: d["date_service_accessed"]
                .astype(str)
                .map(lambda x: x + ":00" if len(x) == 16 else x)
                .map(parse_date),
            service_used=lambda d: d["service_used"].str.strip(),
            service_name=lambda d: d["service_name"].str.strip(),
        )
        .drop(columns=["Name"])
    )

    # learners on these programmes are employed - excluded from indicator 7
    df = df.loc[~df["service_used"].isin(COURSERA_DROP_SERVICES)]

    # pattern-based cohort overrides
    df["cohort_name"] = pd.NA
    df.loc[df["service_used"].str.contains("Pathway", case=False), "cohort_name"] = \
        "SAP Program(Not Listed on Retool)"
    df.loc[df["service_used"].str.contains("Accenture", case=False), "cohort_name"] = \
        "Accenture: Skill Advancement"

    # every remaining programme must be in the cohort map
    unmapped = (set(df.loc[df["cohort_name"].isna(), "service_used"])
                - set(COURSERA_COHORT_MAP))
    checks.add(
        "load", "coursera: all programmes mapped to a cohort", not unmapped,
        "Add these Program Names to COURSERA_COHORT_MAP in CONFIG "
        "(or to COURSERA_DROP_SERVICES if the learners are employed): "
        + "; ".join(sorted(unmapped)),
    )

    mask = df["cohort_name"].isna()
    df.loc[mask, "cohort_name"] = df.loc[mask, "service_used"].map(COURSERA_COHORT_MAP)
    return df

In [ ]:
def load_lxcheckins(cfg):
    """LX coach check-ins (Excel). Emails arrive comma-separated; the template
    headers are verbose free text, renamed to the standard contract here."""
    return (
        pd.read_excel(cfg["path"], usecols=range(2, 8), skiprows=1, sheet_name=0)
        .assign(umuzi_email=lambda d: d["Comma separated list of emails"].str.split(","))
        .drop(columns=["Comma separated list of emails",
                       "If useful for person compiing the data"])
        .explode("umuzi_email", ignore_index=True)
        .assign(cohort_name=lambda d:
                d["Useful for person compiing the data - IF all who attended from same cohort"].str.strip())
        .drop(columns=["Useful for person compiing the data - IF all who attended from same cohort"])
        .rename(columns={
            "type of support service used": "service_used",
            " - type of connection session": "service_name",
            "format DD/MM/YYYY - if only month and year known, put DD as 01": "date_service_accessed",
        })
        .assign(date_service_accessed=lambda d: d["date_service_accessed"].map(parse_date))
        .dropna(subset=["umuzi_email"])
    )

In [ ]:
def load_placements(cfg):
    """Launch Lab placements (Excel, two sheets with different layouts).
    Emails may be separated by commas or semicolons."""
    def read_sheet(sheet_name, usecols):
        return (
            pd.read_excel(
                io=cfg["path"], sheet_name=sheet_name, usecols=usecols, skiprows=2,
                names=["umuzi_email", "cohort_name", "service_used",
                       "service_name", "date_service_accessed"],
            )
            .assign(umuzi_email=lambda d: d["umuzi_email"].str.split(r"\s*[,;]\s*"))
            .explode("umuzi_email", ignore_index=True)
            .assign(
                cohort_name=lambda d: d["cohort_name"].str.strip(),
                date_service_accessed=lambda d: d["date_service_accessed"].map(parse_date),
            )
            .dropna(subset=["umuzi_email"])
        )

    return pd.concat(
        [read_sheet(0, [1, 2, 4, 5, 6]), read_sheet(1, [2, 3, 5, 6, 7])],
        ignore_index=True,
    )

In [ ]:
def load_sap(cfg):
    """SAP support service template (Excel, connection sessions sheet).
    Emails arrive comma-separated."""
    return (
        pd.read_excel(
            io=cfg["path"], sheet_name=1, skiprows=2, usecols=[2, 3, 5, 6, 7],
            names=["umuzi_email", "cohort_name", "service_used",
                   "service_name", "date_service_accessed"],
        )
        .assign(
            umuzi_email=lambda d: d["umuzi_email"].str.split(","),
            date_service_accessed=lambda d: d["date_service_accessed"].map(parse_date),
        )
        .explode("umuzi_email", ignore_index=True)
        .dropna(subset=["umuzi_email"])
    )

In [ ]:
def load_xpl(cfg):
    """Y2 XPL support service (Excel, sheets 3 and 4 of the workbook).
    Template headers are verbose free text, renamed to the contract here."""
    frames = []
    for sheet_index in range(2, 4):
        frames.append(
            pd.read_excel(io=cfg["path"], sheet_name=sheet_index,
                          skiprows=1, usecols=[1, 2, 4, 5, 6])
            .dropna(subset="personal or umuzi - we will check against both")
        )
    return (
        pd.concat(frames, ignore_index=True)
        .rename(columns={
            "personal or umuzi - we will check against both": "umuzi_email",
            "Useful for person compiing the data": "cohort_name",
            "type of support service used": "service_used",
            "course name or module name if applicable to support service accessed": "service_name",
            "format DD/MM/YYYY - if only month and year known, put DD as 01": "date_service_accessed",
        })
        .assign(date_service_accessed=lambda d: d["date_service_accessed"].map(parse_date))
    )


LOADERS = {
    "heart": load_heart,
    "bg": load_bg,
    "coursera": load_coursera,
    "lxcheckins": load_lxcheckins,
    "placements": load_placements,
    "sap": load_sap,
    "xpl": load_xpl,
}

## 5. Load, validate, and combine sources

Each loaded source is checked against the loader contract before it is allowed
into the combined frame: required columns present, at least one row, and every
date parsed. Unparseable dates are shown with their raw values so the template
owner can be told exactly which cells to fix.

In [ ]:
frames = []

for name, cfg in SOURCES.items():
    if not cfg["enabled"]:
        print(f"[skip] {name} (disabled in CONFIG)")
        continue

    df = LOADERS[name](cfg)
    df["source"] = name

    missing_cols = LOADER_COLUMNS - set(df.columns)
    checks.add("load", f"{name}: loader contract columns present", not missing_cols,
               f"Loader returned frame without: {sorted(missing_cols)}")

    checks.add("load", f"{name}: file contains rows", len(df) > 0,
               "Source file loaded 0 rows. Check the path and skiprows in CONFIG, "
               "or disable the source if there was genuinely no activity.",
               level="warn")

    bad_dates = df.loc[df["date_service_accessed"].isna()]
    checks.add("load", f"{name}: all service dates parsed", bad_dates.empty,
               f"{len(bad_dates)} row(s) have unparseable dates. Ask the template "
               "owner to correct these cells (expected DD/MM/YYYY):",
               offenders=bad_dates.head(15))

    if cfg.get("filter_to_period"):
        before = len(df)
        df = df.loc[df["date_service_accessed"].between(
            REPORTING_PERIOD_START, REPORTING_PERIOD_END)]
        print(f"[trim] {name}: {before} -> {len(df)} rows within "
              f"{REPORTING_PERIOD_START.date()}..{REPORTING_PERIOD_END.date()}")

    frames.append(df)
    print(f"[ok]   {name}: {len(df)} rows")

checks.halt_if_failed("load")

data = pd.concat(frames, ignore_index=True)[
    ["source", "umuzi_email", "cohort_name", "service_used",
     "service_name", "date_service_accessed"]
]
print(f"\nCombined: {data.shape[0]} rows from {len(frames)} sources")

## 6. Email canonicalization and match review

Order of operations:

1. Normalize (strip and lowercase).
2. Apply `EMAIL_CORRECTIONS` (known typos and substitutions).
3. Remove `KNOWN_UNREPORTABLE_EMAILS` (verified non-citizens, staff), with a count.
4. Remap personal emails to canonical umuzi emails via `db_fields`.
5. Anything still unmatched produces a review table. With
   `UNMATCHED_EMAIL_POLICY = "halt"` the run stops here until each email is
   either corrected, confirmed unreportable, or added to the database.

In [ ]:
data["umuzi_email"] = normalize_emails(data["umuzi_email"])
data["umuzi_email"] = data["umuzi_email"].replace(EMAIL_CORRECTIONS)

unreportable_mask = data["umuzi_email"].isin(KNOWN_UNREPORTABLE_EMAILS)
if unreportable_mask.any():
    removed = (data.loc[unreportable_mask]
               .groupby("source")["umuzi_email"].agg(["count", "nunique"]))
    print("Removed rows for known-unreportable learners (see CONFIG):")
    display(removed)
data = data.loc[~unreportable_mask].copy()

# personal -> canonical umuzi email
data["umuzi_email"] = data["umuzi_email"].map(lambda x: emailMap.get(x, x))

# review anything that still cannot be matched to the database
unmatched = data.loc[~data["umuzi_email"].isin(set(rich["umuzi_email"]))]

if unmatched.empty:
    print("All emails matched to the database.")
else:
    review = (unmatched.groupby(["source", "umuzi_email"])
              .size().rename("rows").reset_index()
              .sort_values(["source", "umuzi_email"]))
    print(f"{review.shape[0]} email(s) across {unmatched['source'].nunique()} "
          "source(s) could not be matched to db_fields:")
    display(review)
    print(
        "For each email above, do ONE of the following in CONFIG, then re-run:\n"
        "  1. Typo or personal address with a known canonical form "
        "-> add to EMAIL_CORRECTIONS\n"
        "  2. Verified non-SA citizen, staff, or otherwise not reportable "
        "-> add to KNOWN_UNREPORTABLE_EMAILS\n"
        "  3. Learner genuinely missing from the database "
        "-> add them to the database and re-export db_fields.csv"
    )
    if UNMATCHED_EMAIL_POLICY == "halt":
        raise AssertionError(
            f"{review.shape[0]} unmatched email(s). "
            "Resolve using the instructions above, or set "
            "UNMATCHED_EMAIL_POLICY='drop' to knowingly exclude them."
        )
    print(f"\nUNMATCHED_EMAIL_POLICY='drop': excluding {len(unmatched)} row(s).")
    data = data.loc[data["umuzi_email"].isin(set(rich["umuzi_email"]))].copy()

## 7. Enrichment merge

Left-join on the canonical `umuzi_email`. Because Section 3 guarantees
`umuzi_email` is unique in `db_fields`, the row count must be identical before
and after — any change means fan-out and the run stops.

In [ ]:
premerge_rows = data.shape[0]

data = data.merge(rich.drop(columns=["email"]), on="umuzi_email", how="left")

checks.add("merge", "row count unchanged by merge",
           premerge_rows == data.shape[0],
           f"Expected {premerge_rows} rows, got {data.shape[0]}. Duplicate "
           "umuzi_email values in db_fields.csv are fanning out the join.")

null_ids = data.loc[data["learner_id"].isna()]
checks.add("merge", "every row received a learner_id", null_ids.empty,
           "Rows matched an email but got no learner_id; db_fields.csv has "
           "gaps for these learners:",
           offenders=null_ids[["source", "umuzi_email"]].drop_duplicates())

null_dob = data.loc[data["date_of_birth"].isna()]
checks.add("merge", "every row has a date_of_birth", null_dob.empty,
           "Missing dates of birth mean age and age_range will be blank for "
           "these learners; fix in the database if possible:",
           offenders=null_dob[["source", "umuzi_email"]].drop_duplicates(),
           level="warn")

checks.halt_if_failed("merge")

data["learner_id"] = data["learner_id"].astype("int")
data["id_number"] = data["id_number"].str.rjust(13, "0")
print(f"Enriched: {data.shape[0]} rows")

## 8. Derived fields

`month_of_service_accessed` takes both month and year from the service date
itself — the old xpl, lxcheckins, and placements notebooks hardcoded the year.

In [ ]:
data["age_service_accessed"] = (
    pd.to_datetime(data["date_service_accessed"]) - pd.to_datetime(data["date_of_birth"])
).dt.days // 365

data["age_range"] = pd.cut(
    x=data["age_service_accessed"],
    bins=[-np.inf, 17, 25, 35, np.inf],
    labels=["17 and below", "18-25", "26-35", "36+"],
)

data["month_of_service_accessed"] = (
    data["date_service_accessed"].dt.month_name()
    + " "
    + data["date_service_accessed"].dt.year.astype(str)
)

## 9. Final validation suite

Runs against the finished dataset immediately before export. Errors halt the
run; warnings print their offending rows but allow export. The full PASS/WARN/FAIL
table is displayed either way, so a reviewer can see at a glance what was checked.

In [ ]:
final = data[["source"] + FINAL_COLUMNS].copy()

# schema
checks.add("final", "output schema matches IDC template",
           list(final.columns[1:]) == FINAL_COLUMNS,
           f"Column mismatch. Got: {list(final.columns[1:])}")

# dates
future = final.loc[final["date_service_accessed"] > pd.Timestamp.today()]
checks.add("final", "no service dates in the future", future.empty,
           "Future-dated rows almost always mean a day/month swap in the "
           "source template:",
           offenders=future[["source", "umuzi_email", "date_service_accessed"]])

outside = final.loc[~final["date_service_accessed"].between(
    REPORTING_PERIOD_START, REPORTING_PERIOD_END)]
checks.add("final", "all service dates within reporting period", outside.empty,
           f"{len(outside)} row(s) outside "
           f"{REPORTING_PERIOD_START.date()}..{REPORTING_PERIOD_END.date()}. "
           "Legitimate late submissions can stay; stale copies of last month's "
           "template should be removed at source:",
           offenders=(outside.groupby(["source",
                       outside["date_service_accessed"].dt.to_period("M").rename("month")])
                      .size().rename("rows").reset_index()),
           level="warn")

# identifiers
bad_id = final.loc[final["id_number"].notna() & (final["id_number"].str.len() != 13)]
checks.add("final", "id_number is 13 digits", bad_id.empty,
           "SA ID numbers must be 13 digits after zero-padding:",
           offenders=bad_id[["source", "umuzi_email", "id_number"]].drop_duplicates())

null_id = final.loc[final["id_number"].isna()]
checks.add("final", "id_number present for all rows", null_id.empty,
           "Rows without an ID number cannot be verified by the funder:",
           offenders=null_id[["source", "umuzi_email"]].drop_duplicates(),
           level="warn")

# duplicates
dup_keys = ["learner_id", "service_used", "service_name", "date_service_accessed"]
dups = final.loc[final.duplicated(subset=dup_keys, keep=False)].sort_values(dup_keys)
checks.add("final", "no duplicate service records", dups.empty,
           "Identical learner + service + date combinations usually mean the "
           "same session was captured twice (possibly in two different "
           "templates). Verify before submitting:",
           offenders=dups[["source"] + dup_keys], level="warn")

# age sanity
bad_age = final.loc[final["age_service_accessed"].notna()
                    & ~final["age_service_accessed"].between(15, 80)]
checks.add("final", "ages fall within a plausible range (15-80)", bad_age.empty,
           "Implausible ages usually mean a wrong date_of_birth in the database:",
           offenders=bad_age[["source", "umuzi_email", "date_of_birth",
                              "age_service_accessed"]].drop_duplicates(),
           level="warn")

null_cohort = final.loc[final["cohort_name"].isna() | (final["cohort_name"].astype(str).str.strip() == "")]
checks.add("final", "cohort_name present for all rows", null_cohort.empty,
           "Rows without a cohort:",
           offenders=null_cohort[["source", "umuzi_email"]].drop_duplicates(),
           level="warn")

print("\nFull validation summary:")
display(checks.summary())

checks.halt_if_failed()

## 10. Export

A single overwrite write with header — no append, no run-order dependency,
and re-running the notebook can never duplicate rows. The `source` column is
internal and is not exported.

In [ ]:
final.drop(columns=["source"]).to_csv(SINK_PATH, index=False)

print(f"Wrote {final.shape[0]} rows to {SINK_PATH}\n")

summary = (final.groupby(["source", "month_of_service_accessed"])
           .size().rename("rows").reset_index()
           .pivot(index="source", columns="month_of_service_accessed", values="rows")
           .fillna(0).astype(int))
display(summary)

print("\nRows per service type:")
display(final.groupby(["source", "service_used"]).size().rename("rows").reset_index())